# Figure 1: motivation and illustration of BASC

**(a)** Empirical self-consistency (TT-SC) and BASC-L (k=2) confidence both track
answer correctness, pooled over the 36 model/dataset combinations.
**(b, c)** A synthetic illustration of Stage 2: binary isotonic regression on
noisy binary agreement observations recovers a monotone approximation to
E[S | U] and recalibrates the score.

Input: `results/main.csv` (`scripts/experiments/run_main.py`).
Output: `figures/basc_method_figure.png`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression

from basc.stage2 import BinaryIsotonic
from paper_utils import FIGURES_DIR, RESULTS_DIR, pooled_reliability, promote_semantic

plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 14,
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

PALETTE = plt.rcParams['axes.prop_cycle'].by_key()['color']
BASC_C, RAW_C = "#C44E52", "#8C8C8C"
INK  = plt.rcParams['axes.labelcolor']
GRID = plt.rcParams.get('grid.color', '#cccccc')

def style_axes(ax, xlabel=None, ylabel=None):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.7); ax.set_axisbelow(True)
    if xlabel is not None: ax.set_xlabel(xlabel, color=INK, fontsize=13)
    if ylabel is not None: ax.set_ylabel(ylabel, color=INK, fontsize=13)
    return ax

In [ ]:
# Panel (a): reliability pooled over every (model, dataset, seed) row.
df = promote_semantic(pd.read_csv(RESULTS_DIR / 'main.csv'))

LEFT_METHODS = [
    ('oracle_sc_k100', 'Observed', RAW_C),
    ('basc_k2',        'BASC-L',   BASC_C),
]

In [ ]:
# Panels (b, c): synthetic data. Latent probability pi(u) = u^2, so the raw
# score U is overconfident; B | U ~ Bernoulli(pi(U)).
rng = np.random.default_rng(0)

def true_pi(u):
    return u ** 2

N_ISO, N_TEST = 300, 5000
U_iso_syn  = rng.uniform(0.02, 0.98, size=N_ISO)
B_iso_syn  = rng.binomial(1, true_pi(U_iso_syn))
U_test_syn = rng.uniform(0.02, 0.98, size=N_TEST)
Y_test_syn = rng.binomial(1, true_pi(U_test_syn))

# The estimator used by BASC's Stage 2.
_iso_sk = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
_iso_sk.fit(U_iso_syn, B_iso_syn)
iso_syn = BinaryIsotonic(iso=_iso_sk, n_train=N_ISO, n_knots=int(len(_iso_sk.X_thresholds_)))
cal_test_syn = iso_syn.predict(U_test_syn)

def reliability(scores, correct, n_bins=12):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(scores, edges[1:-1]), 0, n_bins - 1)
    counts = np.zeros(n_bins, int)
    mp = np.full(n_bins, np.nan); mc = np.full(n_bins, np.nan)
    for b in range(n_bins):
        m = idx == b
        counts[b] = int(m.sum())
        if counts[b]:
            mp[b] = scores[m].mean()
            mc[b] = correct[m].mean()
    return counts, mp, mc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))

# (a) empirical aggregate reliability
ax = axes[0]
ax.plot([0, 1], [0, 1], ls='--', color='k', lw=1.2, zorder=1, label=r"$x=y$ (perfect)")
for method, label, color in LEFT_METHODS:
    counts, mp, mc = pooled_reliability(df, method)
    ok = counts > 0
    ax.plot(mp[ok], mc[ok], marker='o', color=color, lw=2.2, markersize=6,
            markeredgecolor='white', markeredgewidth=0.8, label=label, zorder=3)
style_axes(ax, xlabel='Self-consistency / Confidence', ylabel='Accuracy')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('(a) Self-consistency vs. Accuracy', color=INK)
ax.legend(loc='upper left', frameon=False, labelcolor=INK, fontsize=12)

# (b) Stage 2 fit on binary agreements
ax = axes[1]
ax.scatter(U_iso_syn[B_iso_syn == 0], np.zeros((B_iso_syn == 0).sum()),
           s=50, color=PALETTE[0], alpha=0.35, edgecolors='white', linewidths=0.3,
           label=r'$B_i = 0$', zorder=2)
ax.scatter(U_iso_syn[B_iso_syn == 1], np.ones((B_iso_syn == 1).sum()),
           s=50, color=PALETTE[1], alpha=0.35, edgecolors='white', linewidths=0.3,
           label=r'$B_i = 1$', zorder=2)
xs = np.linspace(0, 1, 1000)
ax.plot(xs, iso_syn.predict(xs), color=INK, lw=2.4, alpha=0.5,
        label=r'$\hat{g}(U)$ - Iso. Reg.', zorder=4)
ax.plot(xs, true_pi(xs), ls='--', color='k', lw=1.6, label=r'$\mathbb{E}[S\mid U]$', zorder=3)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
ax.set_xlim(0, 1); ax.set_ylim(-0.1, 1.1)
ax.set_xlabel(r'Score  $U$', color=INK, fontsize=13)
ax.set_ylabel(r'Agreement / Probability', color=INK, fontsize=13)
ax.set_yticks([0, 0.5, 1])
ax.set_title('(b) Learning from Binary Agreement', color=INK)
ax.legend(loc='center left', frameon=False, labelcolor=INK, fontsize=12)

# (c) synthetic reliability before / after Stage 2
ax = axes[2]
ax.plot([0, 1], [0, 1], ls='--', color='k', lw=1.2, zorder=1, label=r"$x=y$ (perfect)")
for scores, color, lab in [(U_test_syn, RAW_C, 'Before'), (cal_test_syn, BASC_C, 'After')]:
    counts, mp, mc = reliability(scores, Y_test_syn)
    ok = counts > 0
    ax.plot(mp[ok], mc[ok], marker='o', color=color, lw=2.2, markersize=6,
            markeredgecolor='white', markeredgewidth=0.8, label=lab, zorder=3)
style_axes(ax, xlabel='Predicted Self-Consistency', ylabel='Latent Self-Consistency')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('(c) Recalibration to Latent SC', color=INK)
ax.legend(loc='upper left', frameon=False, labelcolor=INK, fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'basc_method_figure.png')
plt.show()